In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import os

# Task 1: Write your code here:
csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt

def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
df = df.drop('Order_ID', axis = 1)
df

In [ ]:
df.info()

In [ ]:
# Task 2: Write your code here:
catag_cols = ['Weather', 'Traffic_Level', 'Time_of_Day']
for col in catag_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

num_cols  = ['Courier_Experience_yrs', 'Delivery_Time']
for col in num_cols:
    df[col] = df[col].fillna(df[col].mean())

In [ ]:
# Task 3: Write your code here:
duplicates = df.duplicated().sum()
print(f"Duplicates: {duplicates}")


In [ ]:
df = df.drop_duplicates()

In [ ]:
# Task 4: Write your code here:

categorical_cols = df.select_dtypes(include=["object"]).columns
for col in categorical_cols:
  print(df[col].unique())

In [ ]:
#use label encoder for coloumn: Traffic_Level --> better for ordinal data
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['Traffic_Level'] = le.fit_transform(df['Traffic_Level'])
df


In [ ]:
#one hot encoder for other coloumns
from sklearn.preprocessing import OneHotEncoder
oe = OneHotEncoder(sparse_output=False)
for col in categorical_cols:
  encoded = le.fit_transform(df[col])
  df[col] = encoded

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler
scaler_cols = df.drop('Delivery_Time', axis = 1).columns

standard_scaler = StandardScaler()
df[scaler_cols] = standard_scaler.fit_transform(df[scaler_cols])
df

In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Write your code here:
X = df.drop('Delivery_Time', axis = 1)
y = df['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
model = RandomForestRegressor(n_estimators=80, max_depth= 5)

for fold_idx, (train_index, test_index) in enumerate(kfold.split(X)):
  print(f"\nFold {fold_idx + 1}/{5}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  model.fit(X_train, y_train)

  y_pred = model.predict(X_test)

model_mae = mean_absolute_error(y_test, y_pred)
print(model_mae)

In [ ]:
# Task 1: Write your code here:
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
})
importance = importance.sort_values('importance', ascending=True).tail(10)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'])
plt.title('Top 10 Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

plt.figure(figsize=(10, 6))
plt.hist(y_pred, bins=30, edgecolor='black')
plt.title('Distribution of Predictions')
plt.xlabel('Predicted Emission')
plt.ylabel('Count')
plt.show()

In [ ]:
!pip install catboost

In [ ]:
# Task Bonus: Write your code here:
from catboost import CatBoostRegressor

models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=150),
  "CatBoost": CatBoostRegressor(verbose=0, n_estimators=150, max_depth= 5)
}
all_results = {}

for name in models:
  all_results[name] = {'mse': [], 'rmse': [], 'r2': []}


In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{5}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mse = sklearn_mse(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    # Store results
    all_results[model_name]["mse"].append(mse)
    all_results[model_name]["rmse"].append(rmse)
    all_results[model_name]["r2"].append(r2)

In [ ]:
for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  MSE:  {np.mean(all_results[model_name]['mse']):.4f}") # value closer to 0 --> predictions are very close to actual values
  print(f"  RMSE: {np.mean(all_results[model_name]['rmse']):.4f}")  # 0 perfect predictions and lower is better
  print(f"  R2:    {np.mean(all_results[model_name]['r2']):.4f}")